<a href="https://colab.research.google.com/github/nalinkai/Data-Science-Project-Lifecycle/blob/Dev/XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

!pip install xgboost -q
import xgboost as xgb
from xgboost import XGBClassifier

print("✅ Libraries loaded - Ready for XGBoost training")

✅ Libraries loaded - Ready for XGBoost training


In [ ]:
print("="*60)
print("LOADING RAW DATASETS")
print("="*60)

# Load the three files (use your original paths)
train_df = pd.read_csv('/content/drive/MyDrive/Data Science Life Cycle Coursework/Hotel-A-train_finalized.csv')
val_df   = pd.read_csv('/content/drive/MyDrive/Data Science Life Cycle Coursework/Hotel-A-validation.csv')
test_df  = pd.read_csv('/content/drive/MyDrive/Data Science Life Cycle Coursework/Hotel-A-test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Val shape  : {val_df.shape}")
print(f"Test shape : {test_df.shape}")

LOADING RAW DATASETS
Train shape: (26989, 55)
Val shape  : (2749, 24)
Test shape : (4318, 23)


In [ ]:
print("="*60)
print("REMOVING LEAKAGE FEATURES")
print("="*60)

# List of columns that leak target information or are derived from future data
leakage_features = [
    'Reservation_Status_encoded', 'total_cost', 'discounted_rate',
    'previous_stays_cancelled_ratio', 'cancellation_risk',
    'price_per_person', 'lead_time_price_interaction'
]

# Drop leakage from all three datasets if they exist
for df in [train_df, val_df, test_df]:
    cols_to_drop = [col for col in leakage_features if col in df.columns]
    df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print("Leakage features removed.")
print(f"Remaining columns in train: {train_df.shape[1]}")

REMOVING LEAKAGE FEATURES
Leakage features removed.
Remaining columns in train: 47


In [ ]:
print("="*60)
print("APPLYING CONSISTENT PREPROCESSING")
print("="*60)

def preprocess_hotel_data(df, is_train=True, target_col='Reservation_Status'):
    df = df.copy()

    # 1. Basic cleaning
    if target_col in df.columns:
        df[target_col] = df[target_col].str.lower()

    # 2. Feature Engineering (same for all splits)
    if 'Expected_checkin' in df.columns and 'Expected_checkout' in df.columns:
        df['lead_time_days'] = (pd.to_datetime(df['Expected_checkin']) -
                                pd.to_datetime(df['Expected_checkout'])).dt.days.abs()

    if 'Adults' in df.columns and 'Children' in df.columns and 'Babies' in df.columns:
        df['total_guests'] = df['Adults'] + df['Children'] + df['Babies']
        df['is_family'] = (df['Children'] + df['Babies'] > 0).astype(int)

    # 3. Drop original date columns if they exist (to avoid leakage)
    date_cols = ['Expected_checkin', 'Expected_checkout', 'booking_date']
    df.drop(columns=[col for col in date_cols if col in df.columns], inplace=True, errors='ignore')

    return df

# Apply same preprocessing to all three
train_df = preprocess_hotel_data(train_df, is_train=True)
val_df   = preprocess_hotel_data(val_df,   is_train=False)
test_df  = preprocess_hotel_data(test_df,  is_train=False)

print("✅ Consistent preprocessing applied to train/val/test")

APPLYING CONSISTENT PREPROCESSING
✅ Consistent preprocessing applied to train/val/test


In [ ]:
print("="*60)
print("SEPARATING FEATURES AND TARGET")
print("="*60)

target_column = 'Reservation_Status'

# Check if target_column exists in train_df before attempting to separate
if target_column in train_df.columns:
    X_train = train_df.drop(columns=[target_column], errors='ignore')
    y_train = train_df[target_column]
    print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
else:
    print(f"Error: Target column '{target_column}' not found in train_df.")
    print("This is likely because 'Reservation_Status' was prematurely removed in a previous step (e.g., cell ECaOaK8SGl3U which drops 'leakage_features').")
    print("Please ensure the target column is available before separation.")
    # Initialize X_train and y_train to avoid NameError in subsequent cells if this is the chosen path
    X_train = train_df.copy() # No target to drop
    y_train = pd.Series([], dtype='object') # Empty series as placeholder

# Apply similar logic for val_df and test_df
if target_column in val_df.columns:
    X_val = val_df.drop(columns=[target_column], errors='ignore')
    y_val = val_df[target_column]
    print(f"X_val  : {X_val.shape}, y_val  : {y_val.shape}")
else:
    print(f"Warning: Target column '{target_column}' not found in val_df. Assuming it was removed previously.")
    X_val = val_df.copy()
    y_val = pd.Series([], dtype='object')

# X_test might legitimately not have the target column, and y_test is often not available for true test sets
X_test = test_df.drop(columns=[target_column], errors='ignore')
if target_column in test_df.columns:
    y_test = test_df[target_column]
    print(f"X_test : {X_test.shape}, y_test : {y_test.shape}")
else:
    print(f"X_test : {X_test.shape} (target column '{target_column}' not found, as expected for test set without labels)")
    y_test = pd.Series([None] * len(X_test), name=target_column) # Placeholder for missing target in test set


SEPARATING FEATURES AND TARGET
Error: Target column 'Reservation_Status' not found in train_df.
This is likely because 'Reservation_Status' was prematurely removed in a previous step (e.g., cell ECaOaK8SGl3U which drops 'leakage_features').
Please ensure the target column is available before separation.
X_test : (4318, 24) (target column 'Reservation_Status' not found, as expected for test set without labels)
